# 13 - Qwen3-32B Base RAG Evaluation

Heavy Base RAG generation with the best measured retrieval stack: Qwen3-Embedding-8B dense top-30 plus Qwen3-Reranker-8B top-10 context. No fine-tuning adapter is used in this notebook.

In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes peft "sentence-transformers>=5.1.0" faiss-cpu rank-bm25

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from src.generation import run_rag_generation
from src.evaluation_qa import evaluate_generation_predictions

llm_model = 'Qwen/Qwen3-32B'
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
best = config['best_retrieval']
index_root = DRIVE_ROOT / best['index_root']
reranker_model = best['reranker_model']
precomputed_retrieval_csv = DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_top30_qwen3_reranker_8b_predictions_v1.csv'

if not (index_root / 'index_manifest.json').exists():
    raise FileNotFoundError(f'Qwen3 embedding index not found: {index_root}. Run notebook 11 first.')

if not precomputed_retrieval_csv.exists():
    raise FileNotFoundError(f'Precomputed reranker output not found: {precomputed_retrieval_csv}. Run notebook 12 first.')

index_root, reranker_model, precomputed_retrieval_csv

In [ ]:
# Smoke test: run 10 questions before the full 190-question generation.
smoke_predictions = DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_predictions_smoke.csv'
run_rag_generation(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=smoke_predictions,
    output_run_config_json=DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_run_config_smoke.json',
    llm_model=llm_model,
    retriever_mode='dense',
    top_k_context=10,
    candidate_k=30,
    device=device,
    max_new_tokens=512,
    temperature=0.0,
    top_p=1.0,
    max_context_chars=14000,
    input_max_length=8192,
    limit=10,
    load_in_4bit=True,
    adapter_path=None,
    system_name='qwen3_32b_base_rag_best_retrieval_smoke',
    reranker_model=None,
    reranker_batch_size=1,
    precomputed_retrieval_csv=precomputed_retrieval_csv,
)

smoke_summary = evaluate_generation_predictions(
    predictions_csv=smoke_predictions,
    output_eval_csv=DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_eval_smoke.csv',
    output_summary_json=DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_summary_smoke.json',
)
smoke_summary

In [ ]:
# Full run on all 190 benchmark questions.
full_predictions = DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_predictions_v1.csv'
run_rag_generation(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=full_predictions,
    output_run_config_json=DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_run_config_v1.json',
    llm_model=llm_model,
    retriever_mode='dense',
    top_k_context=10,
    candidate_k=30,
    device=device,
    max_new_tokens=512,
    temperature=0.0,
    top_p=1.0,
    max_context_chars=14000,
    input_max_length=8192,
    limit=None,
    load_in_4bit=True,
    adapter_path=None,
    system_name='qwen3_32b_base_rag_best_retrieval',
    reranker_model=None,
    reranker_batch_size=1,
    precomputed_retrieval_csv=precomputed_retrieval_csv,
)

summary = evaluate_generation_predictions(
    predictions_csv=full_predictions,
    output_eval_csv=DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_eval_v1.csv',
    output_summary_json=DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_summary_v1.json',
)
summary